# Model 4: LightGBM (boosting) - raw data to features to scores (Colab)

Owner: Member D. Starts from the **raw trip files**, audits them, builds candidates and features with
Member B's pipeline, trains LightGBM with trip-grouped CV, checks calibration, refits, and scores the student-test trips
with Member C's metrics. Everything you need is produced as a download.

**Upload to Colab (Files panel, drag and drop, into `/content`)**
| File | What it is | Where to get it |
|---|---|---|
| `raw_data.zip` | the 80 train + 10 student-test trips (`raw/train/...`, `raw/student_test/...`) | `colab_upload/raw_data.zip` in the repo folder |
| `pathsnap_bundle.zip` | snapshot of `src/pathsnap` (B's and C's code) + `configs/partitions.yaml` | `colab_upload/pathsnap_bundle.zip` |

Then **Runtime > Run all**. CPU is enough. If you are unsure, set `TRIP_LIMIT = 6` in the config cell first for a 2-minute dry run.

**Downloads you get**: `features.parquet` (right after features are built), `features_meta.zip`, and `outputs_lgbm.zip` (model, run JSON, tables, plots).

**Rules followed**: only the 80 training trips are used for fitting, CV and early stopping; student-test trips are scored once at the end;
the notebook refuses to run if any trip outside `configs/partitions.yaml` is present (this blocks evaluator data).

In [ ]:
%pip install -q lightgbm pyarrow

## 1. Config and unpack the uploads

In [ ]:
import os, sys, json, time, zipfile, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

# ---------------- settings you may change ----------------
TRIP_LIMIT = None          # e.g. 6 = dry run on the first 6 trips of each partition
REBUILD_FEATURES = True    # False reuses work/features.parquet if it already exists
TRAIN_OBS_PER_TRIP = None  # None = every GPS point. 250 reproduces the per-trip subsample in C's harness
INNER_VAL_FRAC = 0.15      # share of TRAINING-fold trips held out for early stopping (not the CV fold)
N_FOLDS = 5
SAVE_TO_DRIVE = False      # True also copies the outputs to Google Drive (protects against a closed session)
FEATURES_VERSION = "v0_pipelineB_15feat"   # TODO(team): agree the official label with B
# ---------------------------------------------------------

WORK = Path("/content/work") if IN_COLAB else Path("work_local")
OUT = WORK / "outputs_lgbm"
OUT.mkdir(parents=True, exist_ok=True)


def find_upload(name):
    for base in [Path("."), Path("/content"), Path("/content/drive/MyDrive/ml_hackathon_uploads")]:
        if (base / name).exists():
            return base / name
    return None


NEEDED = ["raw_data.zip", "pathsnap_bundle.zip"]
if any(find_upload(n) is None for n in NEEDED) and IN_COLAB:
    from google.colab import files
    print("Select the missing zip file(s) to upload...")
    files.upload()
missing = [n for n in NEEDED if find_upload(n) is None]
if missing:
    raise FileNotFoundError(f"Upload these to /content first: {missing}")

CODE_DIR, DATA_DIR = WORK / "code", WORK / "data"
if not (CODE_DIR / "COMMIT.txt").exists():
    zipfile.ZipFile(find_upload("pathsnap_bundle.zip")).extractall(CODE_DIR)
if not (DATA_DIR / "raw").exists():
    zipfile.ZipFile(find_upload("raw_data.zip")).extractall(DATA_DIR)

CODE_COMMIT = (CODE_DIR / "COMMIT.txt").read_text().strip()
GIT_SHORT = CODE_COMMIT.split()[0][:7]
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["PYTHONPATH"] = str(CODE_DIR / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Code bundle commit:", CODE_COMMIT)

In [ ]:
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

from pathsnap.features.build import trip_table, FEATS
from pathsnap.candidates.generate import K_CAND, RADIUS
from pathsnap.evaluation.cv import split_by_trip
from pathsnap.evaluation.metrics import compute_candidate_metrics, compute_route_metrics, candidate_log_loss
from pathsnap.evaluation.run_logger import log_run
from pathsnap.utils.seeds import set_seed

cfg = yaml.safe_load((CODE_DIR / "configs" / "partitions.yaml").read_text())
SEED = int(cfg["seed"])
set_seed(SEED)
norm = lambda ids: sorted(str(t).zfill(8) for t in ids)
TRAIN_TRIPS, STUDENT_TRIPS = norm(cfg["train_trips"]), norm(cfg["student_trips"])
print(f"lightgbm {lgb.__version__} | seed {SEED} | candidates K={K_CAND}, radius={RADIUS} m | {len(FEATS)} features")
print("features:", FEATS)

## 2. Partition guard
The raw folders must match `configs/partitions.yaml` exactly. Anything extra (for example evaluator trips) stops the run.

In [ ]:
RAW = DATA_DIR / "raw"
PARTS = {"train": ("train", TRAIN_TRIPS), "student": ("student_test", STUDENT_TRIPS)}

assert not set(TRAIN_TRIPS) & set(STUDENT_TRIPS), "A trip is listed in two partitions"
for part, (folder, expected) in PARTS.items():
    found = sorted(p.name for p in (RAW / folder).iterdir() if p.is_dir())
    extra, absent = sorted(set(found) - set(expected)), sorted(set(expected) - set(found))
    assert not extra, f"{folder}: trips not in partitions.yaml (evaluator data?): {extra[:5]}"
    assert not absent, f"{folder}: trips missing from the upload: {absent[:5]}"
    print(f"{folder}: {len(found)} trips OK")
if (RAW / "evaluator_test").exists():
    raise RuntimeError("evaluator data present in the upload - remove it")

TRIPS = {part: (folder, expected[:TRIP_LIMIT] if TRIP_LIMIT else expected) for part, (folder, expected) in PARTS.items()}
print({p: len(v[1]) for p, v in TRIPS.items()})

## 3. Data audit and cleaning
A scan of the raw files found no NaNs, no out-of-range node or arc indices and strictly increasing timestamps, so this step
**validates and reports** rather than deleting rows. GPS rows are deliberately **not dropped**: the API predicts one arc per GPS row,
so training must see the same rows. Hard problems stop the run. Soft problems (time gaps, self-loop arcs, very long trips) are
written to `data_quality.csv` for the report.

In [ ]:
def audit_trip(part_dir, tid):
    d = Path(part_dir) / tid
    tr = np.loadtxt(d / f"{tid}.track", ndmin=2)
    nd = np.loadtxt(d / f"{tid}.nodes", ndmin=2)
    ar = np.loadtxt(d / f"{tid}.arcs", dtype=int, ndmin=2)
    rt = np.loadtxt(d / f"{tid}.route", dtype=int, ndmin=1).reshape(-1)
    dt = np.diff(tr[:, 2])
    errors = []
    if tr.shape[1] != 3 or nd.shape[1] != 2 or ar.shape[1] != 2:
        errors.append("unexpected column count")
    if not (np.isfinite(tr).all() and np.isfinite(nd).all()):
        errors.append("non-finite coordinates")
    if (np.abs(tr[:, 0]) > 180).any() or (np.abs(tr[:, 1]) > 90).any():
        errors.append("lon/lat out of range (columns swapped?)")
    if (dt <= 0).any():
        errors.append("timestamps not strictly increasing")
    if ar.min() < 0 or ar.max() >= len(nd):
        errors.append("arc references a missing node")
    if len(rt) == 0 or rt.min() < 0 or rt.max() >= len(ar):
        errors.append("route empty or references a missing arc")
    kx = 111.32 * np.cos(np.radians(tr[:, 1].mean()))
    return {
        "n_points": len(tr), "n_nodes": len(nd), "n_arcs": len(ar), "route_len": len(rt),
        "n_gaps_gt_5s": int((dt > 5).sum()), "max_gap_s": float(dt.max()) if len(dt) else 0.0,
        "share_1s_steps": float((dt == 1).mean()) if len(dt) else 1.0,
        "self_loop_arcs": int((ar[:, 0] == ar[:, 1]).sum()),
        "span_km": float(np.hypot(np.ptp(tr[:, 0]) * kx, np.ptp(tr[:, 1]) * 110.54)),
        "errors": "; ".join(errors),
    }


audit_rows = []
for part, (folder, ids) in TRIPS.items():
    for tid in ids:
        audit_rows.append({"trip_id": tid, "partition": part, **audit_trip(RAW / folder, tid)})
audit = pd.DataFrame(audit_rows)
audit.to_csv(OUT / "data_quality.csv", index=False)

bad = audit[audit.errors != ""]
assert bad.empty, f"Hard data problems, fix before training:\n{bad[['trip_id', 'errors']]}"
print("Hard checks passed for", len(audit), "trips. No rows were removed.")
display(audit.drop(columns="errors").describe().loc[["min", "50%", "max"]].T)
print("trips with time gaps > 5 s:", int((audit.n_gaps_gt_5s > 0).sum()),
      "| trips with self-loop arcs:", int((audit.self_loop_arcs > 0).sum()))

## 4. Candidates, features and labels (Member B's pipeline)
For each GPS point: up to `K` nearest arcs within `RADIUS` m, 15 features per (point, candidate), and `label = 1` when the candidate is the
reference-route arc aligned to that point (monotone DP alignment). The table follows the data contract in `.agents/context/data_contract.md`
(`src_node` and `dst_node` are extra id columns that C's metrics need; they are **not** model features).

In [ ]:
from tqdm.auto import tqdm

FEATURES_PATH = WORK / "features.parquet"
ID_COLS = ["trip_id", "partition", "point_idx", "candidate_arc_id", "label", "src_node", "dst_node"]


def trip_frame(part, folder, tid):
    t = trip_table(str(RAW / folder), tid, with_route=True)
    assert "y" in t, f"trip {tid}: no labels (route missing or empty)"
    arcs, ri, a, y = t["net"].arcs, t["ri"], t["arc"], t["y"]
    f = pd.DataFrame(t["X"], columns=FEATS)
    f.insert(0, "dst_node", arcs[a, 1].astype(np.int32))
    f.insert(0, "src_node", arcs[a, 0].astype(np.int32))
    f.insert(0, "label", y.astype(np.int8))
    f.insert(0, "candidate_arc_id", a.astype(np.int32))
    f.insert(0, "point_idx", ri.astype(np.int32))
    f.insert(0, "partition", part)
    f.insert(0, "trip_id", tid)
    n_pts = len(t["P"])
    summ = {
        "trip_id": tid, "partition": part, "n_points": n_pts, "n_candidate_rows": len(f),
        "points_with_candidates": int(len(np.unique(ri))), "points_with_true_candidate": int(y.sum()),
        "candidate_miss_rate": 1.0 - float(y.sum()) / n_pts,
        "label_align_cost_median_m": float(np.median(t["ref_cost"])), "label_align_cost_p95_m": float(np.percentile(t["ref_cost"], 95)),
    }
    return f, summ


if FEATURES_PATH.exists() and not REBUILD_FEATURES:
    feat = pd.read_parquet(FEATURES_PATH)
    summ_df = pd.read_csv(OUT / "candidate_summary.csv")
    print("Reusing", FEATURES_PATH)
else:
    frames, summaries, t0 = [], [], time.time()
    jobs = [(part, folder, tid) for part, (folder, ids) in TRIPS.items() for tid in ids]
    for part, folder, tid in tqdm(jobs, desc="building features"):
        f, s = trip_frame(part, folder, tid)
        frames.append(f); summaries.append(s)
    feat = pd.concat(frames, ignore_index=True)
    summ_df = pd.DataFrame(summaries)
    summ_df.to_csv(OUT / "candidate_summary.csv", index=False)
    print(f"Built {len(feat):,} candidate rows from {len(jobs)} trips in {time.time() - t0:.0f}s")

assert feat[FEATS].notna().all().all(), "NaNs in features"
assert np.isfinite(feat[FEATS].to_numpy()).all(), "inf in features"
print(feat.groupby("partition").agg(trips=("trip_id", "nunique"), rows=("label", "size"), positives=("label", "sum")))
print("positive rate:", round(float(feat.label.mean()), 4))
print("candidate miss rate (train, mean over trips): %.4f | (student): %.4f" % tuple(
    summ_df[summ_df.partition == p].candidate_miss_rate.mean() for p in ["train", "student"]))
print("label alignment cost: median of per-trip medians %.1f m, worst trip p95 %.1f m" % (
    summ_df.label_align_cost_median_m.median(), summ_df.label_align_cost_p95_m.max()))
display(feat.head(3))

## 5. Save and download the feature table

In [ ]:
feat.to_parquet(FEATURES_PATH, index=False)
manifest = {
    "features_version": FEATURES_VERSION, "created": datetime.now().isoformat(timespec="seconds"),
    "code_bundle_commit": CODE_COMMIT, "candidates": {"K": K_CAND, "radius_m": RADIUS},
    "feature_columns": FEATS, "id_columns": ID_COLS,
    "label_rule": "1 if candidate arc equals the reference-route arc aligned to the GPS point (monotone DP, skip <=3)",
    "rows": int(len(feat)), "trips": {p: int(feat[feat.partition == p].trip_id.nunique()) for p in ["train", "student"]},
    "positive_rate": float(feat.label.mean()), "trip_limit": TRIP_LIMIT,
    "dtypes": {c: str(t) for c, t in feat.dtypes.items()},
}
(OUT / "features_manifest.json").write_text(json.dumps(manifest, indent=2))

meta_zip = WORK / "features_meta.zip"
with zipfile.ZipFile(meta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["features_manifest.json", "data_quality.csv", "candidate_summary.csv"]:
        z.write(OUT / name, name)
print(f"features.parquet: {FEATURES_PATH.stat().st_size / 1e6:.1f} MB | {len(feat):,} rows | {len(FEATS)} features")

if IN_COLAB:
    from google.colab import files
    files.download(str(FEATURES_PATH))      # allow multiple downloads if the browser asks
    files.download(str(meta_zip))

## 6. Train/student split and trip-grouped folds
Fitting, CV and early stopping use only the `train` partition. Folds come from C's `split_by_trip` (GroupKFold by trip, with a leakage assertion).

In [ ]:
def subsample_points(frame, n_per_trip, seed):
    # keep whole candidate groups: sample GPS points, never single rows
    rng = np.random.default_rng(seed)
    keep = np.zeros(len(frame), dtype=bool)
    for tid, idx in frame.groupby("trip_id").indices.items():
        pts = frame.point_idx.values[idx]
        uniq = np.unique(pts)
        keep[idx] = np.isin(pts, rng.choice(uniq, n_per_trip, replace=False)) if len(uniq) > n_per_trip else True
    return frame[keep]


train_df = feat[feat.partition == "train"].reset_index(drop=True)
student_df = feat[feat.partition == "student"].reset_index(drop=True)
if TRAIN_OBS_PER_TRIP:
    train_df = subsample_points(train_df, TRAIN_OBS_PER_TRIP, SEED).reset_index(drop=True)
assert set(train_df.trip_id).isdisjoint(student_df.trip_id) and len(train_df) and len(student_df)

X_train = train_df[FEATS].to_numpy(np.float32)
y_train = train_df.label.to_numpy()
groups = train_df.trip_id.to_numpy()
X_student = student_df[FEATS].to_numpy(np.float32)
y_student = student_df.label.to_numpy()

folds = [(tr, va, k) for tr, va, k in split_by_trip(X_train, y_train, groups, n_splits=N_FOLDS)]
for tr, va, k in folds:
    print(f"fold {k}: train {len(tr):,} rows / {len(np.unique(groups[tr]))} trips | val {len(va):,} rows / {len(np.unique(groups[va]))} trips")
print(f"train rows {len(train_df):,} (positive rate {y_train.mean():.4f}) | student rows {len(student_df):,}")

## 7. Helpers (Top-1, early-stopping split, calibration)

In [ ]:
def top1_fast(frame, p):
    # Same definition as C's observation_top1_accuracy: the highest-scoring candidate of each (trip, point)
    # must be the true arc; points whose true arc is not among the candidates count as wrong.
    d = pd.DataFrame({"t": frame.trip_id.values, "o": frame.point_idx.values, "p": p, "y": frame.label.values})
    return float(d.loc[d.groupby(["t", "o"], sort=False)["p"].idxmax(), "y"].mean())


def make_metas(frame, p):
    metas, chunks = [], []
    for tid, idx in sorted(frame.groupby("trip_id").indices.items()):
        s = frame.iloc[idx]
        M = np.column_stack([s.point_idx.values, s.candidate_arc_id.values, s.src_node.values, s.dst_node.values])
        metas.append({"M": M, "y": s.label.values}); chunks.append(p[idx])
    return metas, chunks


def inner_trip_split(trip_ids, frac, seed):
    # hold out whole trips (never rows) for early stopping
    rng = np.random.default_rng(seed)
    trips = np.unique(trip_ids); rng.shuffle(trips)
    val = trips[:max(1, int(round(len(trips) * frac)))]
    is_val = np.isin(trip_ids, val)
    return np.where(~is_val)[0], np.where(is_val)[0]


def calibration_check(y, p, trip_ids, n_splits=5):
    # sigmoid / isotonic fitted on out-of-fold probabilities with trip-grouped CV; keep only if log loss improves
    eps = 1e-7
    clip = lambda q: np.clip(q, eps, 1 - eps)
    logit = lambda q: np.log(clip(q) / (1 - clip(q))).reshape(-1, 1)
    rows = {"raw": [], "sigmoid": [], "isotonic": []}
    for tr, va in GroupKFold(n_splits=n_splits).split(p, y, trip_ids):
        rows["raw"].append(candidate_log_loss(y[va], p[va]))
        sig = LogisticRegression(C=1e6).fit(logit(p[tr]), y[tr])
        rows["sigmoid"].append(candidate_log_loss(y[va], sig.predict_proba(logit(p[va]))[:, 1]))
        iso = IsotonicRegression(out_of_bounds="clip", y_min=eps, y_max=1 - eps).fit(p[tr], y[tr])
        rows["isotonic"].append(candidate_log_loss(y[va], iso.predict(p[va])))
    table = pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in rows.items()}, index=["mean_log_loss", "std"]).T
    return table, table.mean_log_loss.idxmin()


def reliability_plot(y, p, title, path, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    pts = [(p[idx == b].mean(), y[idx == b].mean()) for b in range(bins) if (idx == b).sum() >= 50]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], "k--", label="perfect"); ax.plot(*zip(*pts), "o-", label="model")
    ax.set(xlabel="predicted P(correct)", ylabel="observed frequency", title=title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()

## 8. LightGBM hyperparameters (initial, untuned)
Objective: binary log loss. `bagging_freq=1` is needed for `bagging_fraction` to take effect. Early stopping watches log loss on held-out
*training-fold trips*. These values go into `configs/models/lgbm.yaml` and report section 3.2.

In [ ]:
LGB_PARAMS = {
    "objective": "binary", "metric": "binary_logloss", "boosting_type": "gbdt",
    "learning_rate": 0.1, "num_leaves": 31, "max_depth": -1, "min_data_in_leaf": 20,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.0, "lambda_l2": 0.0,
    "seed": SEED, "bagging_seed": SEED, "feature_fraction_seed": SEED, "deterministic": True,
    "num_threads": 0, "verbosity": -1,
}
MAX_ROUNDS, EARLY_STOP = 1000, 30
pd.Series({**LGB_PARAMS, "max_rounds": MAX_ROUNDS, "early_stopping_rounds": EARLY_STOP}).astype(str).to_frame("value")

## 9. Cross-validation (5-fold GroupKFold by trip)

In [ ]:
cv_rows, oof, best_iters, gain = [], np.zeros(len(train_df)), [], np.zeros(len(FEATS))
rng_s = np.random.default_rng(SEED)

for tr, va, k in folds:
    fit_i, mon_i = inner_trip_split(groups[tr], INNER_VAL_FRAC, SEED + k)
    fit_idx, mon_idx = tr[fit_i], tr[mon_i]
    dfit = lgb.Dataset(X_train[fit_idx], label=y_train[fit_idx], feature_name=FEATS)
    dmon = lgb.Dataset(X_train[mon_idx], label=y_train[mon_idx], reference=dfit)

    t0 = time.perf_counter()
    booster = lgb.train(LGB_PARAMS, dfit, num_boost_round=MAX_ROUNDS, valid_sets=[dmon], valid_names=["monitor"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    train_s = time.perf_counter() - t0
    bi = booster.best_iteration or booster.current_iteration()

    t0 = time.perf_counter()
    p_va = booster.predict(X_train[va], num_iteration=bi)
    infer_s = time.perf_counter() - t0
    oof[va] = p_va

    samp = rng_s.choice(fit_idx, size=min(300_000, len(fit_idx)), replace=False)
    m_tr = compute_candidate_metrics(y_train[samp], booster.predict(X_train[samp], num_iteration=bi))
    m = compute_candidate_metrics(y_train[va], p_va)
    cv_rows.append({"fold": k, **m, "top1": top1_fast(train_df.iloc[va], p_va), "best_iter": bi,
                    "train_macro_f1": m_tr["candidate_macro_f1"], "train_log_loss": m_tr["candidate_log_loss"],
                    "train_time_s": train_s, "infer_time_s": infer_s})
    best_iters.append(bi); gain += booster.feature_importance("gain")
    print(f"fold {k}: macroF1={m['candidate_macro_f1']:.4f} logloss={m['candidate_log_loss']:.4f} "
          f"top1={cv_rows[-1]['top1']:.4f} best_iter={bi} train={train_s:.1f}s")

cv_df = pd.DataFrame(cv_rows).set_index("fold")
display(cv_df)

## 10. CV summary, fold stability and overfitting evidence

In [ ]:
display(cv_df.agg(["mean", "std"]).T)
f1m, f1s = cv_df.candidate_macro_f1.mean(), cv_df.candidate_macro_f1.std()
print(f"CV candidate macro F1 = {f1m:.4f} +/- {f1s:.4f} (folds {cv_df.candidate_macro_f1.min():.4f} to {cv_df.candidate_macro_f1.max():.4f})")
print(f"train-minus-CV macro F1 gap: {(cv_df.train_macro_f1 - cv_df.candidate_macro_f1).mean():.4f}")
print(f"CV-minus-train log loss gap: {(cv_df.candidate_log_loss - cv_df.train_log_loss).mean():.4f}")
print(f"total CV training time: {cv_df.train_time_s.sum():.1f}s")

## 11. Feature importance and a simple redundancy check
Gain importance (summed over folds) shows what the model uses. Highly correlated pairs (|r| > 0.95) are listed as candidates for removal.
**Nothing is dropped here**: changing the feature set changes the handoff contract, so it needs a team decision (and B's agreement).

In [ ]:
imp = pd.Series(gain, index=FEATS).sort_values(ascending=False)
imp_df = (imp / imp.sum()).rename("gain_share").to_frame()
display(imp_df)
fig, ax = plt.subplots(figsize=(7, 4.5))
imp_df.sort_values("gain_share").plot.barh(ax=ax, legend=False); ax.set_title("LightGBM feature importance (share of gain)")
fig.tight_layout(); fig.savefig(OUT / "lgbm_feature_importance.png", dpi=150); plt.show()

sample = train_df[FEATS].sample(min(200_000, len(train_df)), random_state=SEED)
corr = sample.corr().abs()
pairs = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(FEATS) for b in FEATS[i + 1:] if corr.loc[a, b] > 0.95]
print("pairs with |r| > 0.95:", pairs or "none")
print("features with ~zero gain:", list(imp_df[imp_df.gain_share < 1e-4].index) or "none")
corr.to_csv(OUT / "feature_correlation.csv")

## 12. Calibration check
Viterbi consumes these probabilities, so log loss matters. Calibrators are fitted with trip-grouped CV on the out-of-fold predictions; keep one only if it lowers log loss.

In [ ]:
cal_table, best_cal = calibration_check(y_train, oof, groups)
display(cal_table)
KEEP_CALIBRATION = best_cal != "raw"
print("lowest log loss:", best_cal, "| keep calibration:", KEEP_CALIBRATION)
reliability_plot(y_train, oof, "LightGBM reliability (out-of-fold, train trips)", OUT / "lgbm_reliability.png")

## 13. Final refit on all 80 training trips, then one student-test evaluation

In [ ]:
n_rounds = int(round(np.mean(best_iters)))
t0 = time.perf_counter()
final = lgb.train(LGB_PARAMS, lgb.Dataset(X_train, label=y_train, feature_name=FEATS), num_boost_round=n_rounds)
final_train_s = time.perf_counter() - t0

t0 = time.perf_counter()
p_stu = final.predict(X_student)
final_infer_s = time.perf_counter() - t0

stu_cand = compute_candidate_metrics(y_student, p_stu)
metas, chunks = make_metas(student_df, p_stu)
stu_route = compute_route_metrics(metas, chunks)
fast = top1_fast(student_df, p_stu)
print("Top-1 check (C's function vs fast version): %.6f vs %.6f -> %s" % (
    stu_route["student_test_top1_accuracy"], fast, "OK" if abs(stu_route["student_test_top1_accuracy"] - fast) < 1e-9 else "MISMATCH"))

stu = pd.Series({**stu_cand, **stu_route, "rounds": n_rounds, "train_time_s": final_train_s, "infer_time_s": final_infer_s},
                name="LightGBM (untuned)").to_frame()
display(stu)
print(f"final rounds={n_rounds} | train {final_train_s:.1f}s | inference on {len(student_df):,} rows {final_infer_s:.3f}s")

## 14. Save model, tables and the run JSON

In [ ]:
for sub in ["artifacts", "results", "experiments/runs"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

final.save_model(str(OUT / "artifacts/lgbm_untuned.txt"))
(OUT / "artifacts/lgbm_feature_names.json").write_text(json.dumps(FEATS, indent=2))
cv_df.to_csv(OUT / "results/lgbm_cv_folds.csv")
imp_df.to_csv(OUT / "results/lgbm_feature_importance.csv")
cal_table.to_csv(OUT / "results/lgbm_calibration.csv")
stu.to_csv(OUT / "results/lgbm_student_test.csv")

run_path = log_run(
    run_id="run_lightgbm_v0", model="LightGBM", family="boosting (leaf-wise gradient boosted trees)", author="Member D",
    config_path="configs/models/lgbm.yaml", seed=SEED, features_version=FEATURES_VERSION,
    cv_macro_f1=f1m, student_top1=stu_route["student_test_top1_accuracy"], student_macro_f1=stu_cand["candidate_macro_f1"],
    log_loss=stu_cand["candidate_log_loss"], train_time_s=final_train_s, infer_time_s=final_infer_s,
    git_commit=GIT_SHORT, output_dir=OUT / "experiments/runs",
    extra_metrics={
        "cv_macro_f1_std": float(f1s), "cv_log_loss_mean": float(cv_df.candidate_log_loss.mean()),
        "cv_top1_mean": float(cv_df.top1.mean()), "cv_train_time_s": float(cv_df.train_time_s.sum()),
        "route_continuity": stu_route["route_continuity"], "candidate_miss_rate": stu_route["candidate_miss_rate"],
        "hyperparameters": {**LGB_PARAMS, "num_boost_round_final": n_rounds, "early_stopping_rounds": EARLY_STOP},
        "calibration": {"lowest": best_cal, "kept": bool(KEEP_CALIBRATION), "mean_log_loss": cal_table.mean_log_loss.to_dict()},
        "n_features": len(FEATS), "train_obs_per_trip": TRAIN_OBS_PER_TRIP, "trip_limit": TRIP_LIMIT,
        "train_rows": int(len(train_df)), "student_rows": int(len(student_df)),
        "code_bundle_commit": CODE_COMMIT,
    },
)
print(Path(run_path).read_text())

## 15. Zip and download the results

In [ ]:
zip_path = WORK / "outputs_lgbm.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob("*")):
        if f.is_file():
            z.write(f, Path("outputs_lgbm") / f.relative_to(OUT))
print("Created", zip_path)
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT)}  ({f.stat().st_size / 1e3:.0f} KB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/ml_hackathon_outputs"); dest.mkdir(parents=True, exist_ok=True)
        for f in [zip_path, meta_zip, FEATURES_PATH]:
            shutil.copy(f, dest / f.name)
        print("Copied to", dest)